# Task 3 — AQE Variant: distance_by_weather

## Overview

This notebook tests AQE using Query 2 (distance_by_weather), as a third data point alongside the main AQE notebook (Query 4) and the monthly_zone_demand variant.

We chose this query because trip counts are very uneven across weather categories, with far more clear weather hours than storms, and we expected that imbalance to give AQE's skew handling something to work with. That expectation turned out to be wrong, as explained in the discussion below.

## Query design

The query is Query 2 exactly as written in the Task 1/2 library, unmodified. Only spark.sql.adaptive.enabled differs between the baseline and optimized runs.

## Methodology

Same as the main AQE notebook: one discarded warm-up run per configuration, five timed repetitions, median reported. Both configurations are measured in one SparkSession, with the setting changed right before each configuration's measurements.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parents[1] / "src"))

from data_intensive_computing.benchmark_helpers import (
    build_spark_session,
    register_delta_tables,
    timed_run,
    capture_explain,
    append_benchmark_record,
    find_project_root,
)

PROJECT_ROOT = find_project_root(Path.cwd().resolve())
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
RESULTS_PATH = PROJECT_ROOT / "artifacts" / "task3_benchmark_results.jsonl"

spark = build_spark_session(app_name="Task3-AQE-DistanceByWeather")
register_delta_tables(spark, PROCESSED_DIR)

:: loading settings :: url = jar:file:/Users/rexchang/Documents/KTH/Courses/26-Fall-P1/Data-Intensive-Computing/Homework/data-intensive-computing/.venv/lib/python3.14/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /Users/rexchang/.ivy2.5.2/cache
The jars for the packages stored in: /Users/rexchang/.ivy2.5.2/jars
io.delta#delta-spark_4.2_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-0ad0d99f-f65b-43bd-8891-ac801f99310b;1.0
	confs: [default]
	found io.delta#delta-spark_4.2_2.13;4.4.0 in central
	found io.delta#delta-storage;4.4.0 in central
	found io.unitycatalog#unitycatalog-client;0.6.0 in central
	found org.slf4j#slf4j-api;2.0.13 in central
	found org.apache.logging.log4j#log4j-slf4j2-impl;2.25.3 in central
	found org.apache.logging.log4j#log4j-api;2.25.3 in central
	found com.google.code.findbugs#jsr305;3.0.2 in central
	found io.unitycatalog#unitycatalog-hadoop;0.6.0 in c

Registered taxi_trips: /Users/rexchang/Documents/KTH/Courses/26-Fall-P1/Data-Intensive-Computing/Homework/data-intensive-computing/data/processed/taxi_trips
Registered weather: /Users/rexchang/Documents/KTH/Courses/26-Fall-P1/Data-Intensive-Computing/Homework/data-intensive-computing/data/processed/weather
Registered air_quality: /Users/rexchang/Documents/KTH/Courses/26-Fall-P1/Data-Intensive-Computing/Homework/data-intensive-computing/data/processed/air_quality
Registered taxi_zones: /Users/rexchang/Documents/KTH/Courses/26-Fall-P1/Data-Intensive-Computing/Homework/data-intensive-computing/data/processed/taxi_zones
Registered integrated_taxi_trips: /Users/rexchang/Documents/KTH/Courses/26-Fall-P1/Data-Intensive-Computing/Homework/data-intensive-computing/data/processed/integrated_taxi_trips


26/09/18 00:02:43 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


## Query

In [2]:
QUERY_SQL = """
WITH classified AS (
    SELECT
        trip_id,
        trip_distance,
        CASE
            WHEN weather_condition_code IN (1, 2) THEN 'clear_or_fair'
            WHEN weather_condition_code IN (3, 4) THEN 'cloudy'
            WHEN weather_condition_code IN (5, 6) THEN 'fog'
            WHEN weather_condition_code IN (7, 8, 9, 10, 11, 17, 18) THEN 'rain'
            WHEN weather_condition_code IN (12, 13, 14, 15, 16, 19, 20, 21, 22) THEN 'snow_or_sleet'
            WHEN weather_condition_code BETWEEN 23 AND 27 THEN 'storm'
            ELSE 'unknown'
        END AS weather_condition
    FROM integrated_taxi_trips
    WHERE weather_match_status = 'exact_hour'
      AND trip_distance IS NOT NULL
)
SELECT
    weather_condition,
    COUNT(trip_id) AS trip_count,
    ROUND(AVG(trip_distance), 3) AS average_trip_distance,
    ROUND(STDDEV_POP(trip_distance), 3) AS trip_distance_stddev
FROM classified
GROUP BY weather_condition
ORDER BY average_trip_distance DESC
"""

## Baseline: AQE disabled

In [3]:
spark.conf.set("spark.sql.adaptive.enabled", "false")

baseline_timing = timed_run(spark, "distance_by_weather_aqe_off", QUERY_SQL, repetitions=5)
print("Baseline (AQE disabled):", baseline_timing)

Baseline (AQE disabled): {'query_name': 'distance_by_weather_aqe_off', 'row_count': 5, 'repetitions': 5, 'all_times_seconds': [0.7287, 0.4041, 0.4077, 0.3357, 0.2974], 'median_seconds': 0.4041, 'min_seconds': 0.2974, 'max_seconds': 0.7287}


In [4]:
baseline_plan = capture_explain(spark.sql(QUERY_SQL), mode="formatted")
baseline_rows = spark.sql(QUERY_SQL).collect()

print("=" * 30, "BASELINE PLAN (AQE disabled)", "=" * 30)
print(baseline_plan)

============================== BASELINE PLAN (AQE disabled) ==============================
== Physical Plan ==
* Sort (9)
+- Exchange (8)
   +- * HashAggregate (7)
      +- Exchange (6)
         +- * HashAggregate (5)
            +- * Project (4)
               +- * Filter (3)
                  +- * ColumnarToRow (2)
                     +- Scan parquet  (1)


(1) Scan parquet 
Output [6]: [trip_distance#102, trip_id#121, weather_condition_code#141, weather_match_status#142, pickup_year#125, pickup_month#126]
Batched: true
Location: PreparedDeltaFileIndex [file:/Users/rexchang/Documents/KTH/Courses/26-Fall-P1/Data-Intensive-Computing/Homework/data-intensive-computing/data/processed/integrated_taxi_trips]
PushedFilters: [IsNotNull(weather_match_status), EqualTo(weather_match_status,exact_hour), IsNotNull(trip_distance)]
ReadSchema: struct<trip_distance:double,trip_id:string,weather_condition_code:int,weather_match_status:string>

(2) ColumnarToRow [codegen id : 1]
Input [6]: [trip_dista

## Optimized: AQE enabled

The plan is captured *after* `timed_run` has already executed this exact
query under AQE, so it reflects the plan Spark actually converged on at
runtime (its `== Final Plan ==` section) rather than only the plan chosen
before any shuffle stage had materialized.

In [5]:
spark.conf.set("spark.sql.adaptive.enabled", "true")

optimized_timing = timed_run(spark, "distance_by_weather_aqe_on", QUERY_SQL, repetitions=5)
print("Optimized (AQE enabled):", optimized_timing)

Optimized (AQE enabled): {'query_name': 'distance_by_weather_aqe_on', 'row_count': 5, 'repetitions': 5, 'all_times_seconds': [0.463, 0.3336, 0.2792, 0.2885, 0.2949], 'median_seconds': 0.2949, 'min_seconds': 0.2792, 'max_seconds': 0.463}


In [6]:
optimized_plan = capture_explain(spark.sql(QUERY_SQL), mode="formatted")
optimized_rows = spark.sql(QUERY_SQL).collect()

print("=" * 30, "OPTIMIZED PLAN (AQE enabled)", "=" * 30)
print(optimized_plan)

============================== OPTIMIZED PLAN (AQE enabled) ==============================
== Physical Plan ==
AdaptiveSparkPlan (9)
+- Sort (8)
   +- Exchange (7)
      +- HashAggregate (6)
         +- Exchange (5)
            +- HashAggregate (4)
               +- Project (3)
                  +- Filter (2)
                     +- Scan parquet  (1)


(1) Scan parquet 
Output [6]: [trip_distance#102, trip_id#121, weather_condition_code#141, weather_match_status#142, pickup_year#125, pickup_month#126]
Batched: true
Location: PreparedDeltaFileIndex [file:/Users/rexchang/Documents/KTH/Courses/26-Fall-P1/Data-Intensive-Computing/Homework/data-intensive-computing/data/processed/integrated_taxi_trips]
PushedFilters: [IsNotNull(weather_match_status), EqualTo(weather_match_status,exact_hour), IsNotNull(trip_distance)]
ReadSchema: struct<trip_distance:double,trip_id:string,weather_condition_code:int,weather_match_status:string>

(2) Filter
Input [6]: [trip_distance#102, trip_id#121, weather_co

## Physical plan comparison

We compared the baseline and optimized plans directly. They are identical: no AQEShuffleRead, no difference between the Final Plan and Initial Plan, and no change in join strategy or partition counts. AQE did not adjust anything for this query.

## Correctness verification

Since both variants run the identical SQL text, results are collected under
each configuration and compared directly.

In [7]:
baseline_set = {tuple(r) for r in baseline_rows}
optimized_set = {tuple(r) for r in optimized_rows}

rows_only_in_before = len(baseline_set - optimized_set)
rows_only_in_after = len(optimized_set - baseline_set)
correctness_verified = (rows_only_in_before == 0) and (rows_only_in_after == 0)

print({
    "correctness_verified": correctness_verified,
    "rows_only_in_before": rows_only_in_before,
    "rows_only_in_after": rows_only_in_after,
})
assert correctness_verified, "AQE on/off produced different results — investigate before recording a result."


{'correctness_verified': True, 'rows_only_in_before': 0, 'rows_only_in_after': 0}


## Benchmark record

In [8]:
speedup = round(baseline_timing["median_seconds"] / optimized_timing["median_seconds"], 2)

has_shuffle_read = "AQEShuffleRead" in optimized_plan
has_final_initial_split = "== Final Plan ==" in optimized_plan and "== Initial Plan ==" in optimized_plan
plans_differ = False
if has_final_initial_split:
    final_section = optimized_plan.split("== Final Plan ==")[1].split("== Initial Plan ==")[0]
    initial_section = optimized_plan.split("== Initial Plan ==")[1]
    plans_differ = final_section.strip() != initial_section.strip()

plan_change_summary = (
    "The optimized plan is wrapped in AdaptiveSparkPlan, absent from the baseline. "
    + ("AQEShuffleRead is present, indicating AQE adjusted shuffle partitions after "
       "observing their actual size. "
       if has_shuffle_read else
       "AQEShuffleRead is not present, indicating no post-shuffle partition "
       "adjustment was triggered at this data scale. ")
    + (f"The Final Plan differs from the Initial Plan, confirming AQE altered "
       f"execution based on runtime statistics."
       if plans_differ else
       "The Final Plan matches the Initial Plan, indicating the static estimates "
       "were already accurate enough that no re-planning occurred for this query "
       "at this data scale.")
)

record = {
    "query_name": "distance_by_weather",
    "technique": "aqe",
    "before_time_sec": baseline_timing["median_seconds"],
    "after_time_sec": optimized_timing["median_seconds"],
    "speedup": speedup,
    "plan_change_summary": plan_change_summary,
    "correctness_verified": correctness_verified,
}

append_benchmark_record(record, RESULTS_PATH)
print(f"Speedup: {speedup}x — appended to {RESULTS_PATH}")
print()
print("plan_change_summary:", plan_change_summary)

Speedup: 1.37x — appended to /Users/rexchang/Documents/KTH/Courses/26-Fall-P1/Data-Intensive-Computing/Homework/data-intensive-computing/artifacts/task3_benchmark_results.jsonl

plan_change_summary: The optimized plan is wrapped in AdaptiveSparkPlan, absent from the baseline. AQEShuffleRead is not present, indicating no post-shuffle partition adjustment was triggered at this data scale. The Final Plan matches the Initial Plan, indicating the static estimates were already accurate enough that no re-planning occurred for this query at this data scale.


## Discussion

Result: 1.37x on the first run. The plan shows no AQEShuffleRead, and the Final Plan matches the Initial Plan, meaning AQE found nothing worth changing.

Our skew hypothesis doesn't actually apply to this query. There is no join here, so AQE's skew-join handling is irrelevant. The final output is also only 6 or 7 rows, one per weather category, so there's nothing small left over for AQE to coalesce either.

We tested the 1.37x result by reversing the measurement order (see aqe_distance_by_weather_reverse.ipynb). With AQE disabled now measured second, it came out faster (1.45x), the opposite of the original result. This confirms the original 1.37x came from measurement order, not from AQE.

Combined with Query 4 and monthly_zone_demand, all three queries we tested showed no real AQE effect at this data scale, each for a different reason.

In [9]:
spark.stop()